# sg-sem: build the data and train the model

The Skill Graph semantic checker (see `SKILL_GRAPH_SEMANTIC_MODEL_SPEC.md` in the repo). This notebook runs on a **Colab GPU** through the VS Code Colab extension:

1. clones the repo and runs our own graph library (Node) to build seed graphs and text jobs,
2. runs an open model **on this GPU** (no API) to write outlines, checker answers, plain-word phrases and gates,
3. assembles the labelled examples, split by graph,
4. fine-tunes the model and reports dev and test metrics against the 0.9 precision bar.
5. scores the hand-written **gold set** (`gold/`), which the model never saw: it decides what ships.

**Run it with `QUICK_LOOK = True` first** (about 15 minutes): it writes a few answers per round so their quality can be checked. Then set it to `False` and run all again; finished answers are kept, so nothing is redone.

Save the notebook when it finishes: its outputs are what gets reviewed.

In [1]:
REPO = "https://github.com/Saad-web-spec/DevScratchPad.git"
BRANCH = "sg-sem"  # the branch with the sg-sem code
GENERATOR = "Qwen/Qwen2.5-7B-Instruct"   # writes the text; runs here on the GPU
BASE = "cross-encoder/ms-marco-MiniLM-L12-v2"  # the encoder sg-sem is fine-tuned from: the same MiniLM-L12-H384 network, already trained on text pairs (the raw microsoft/MiniLM-L12-H384-uncased never was)
EPOCHS = 10                                 # 4 through minilm-5: H2 loss was still falling at the end
QUICK_LOOK = False                         # True: a few answers per round, to check quality first (done; the full data is on Drive)
TRAIN = not QUICK_LOOK                     # training waits for the full data
RUN = "msmarco-1"  # minilm-1: H1 missed buried problems, H2 at chance. minilm-2: H1 flagged honest number ledgers, H2 starved (68 positives). minilm-3: H2 read the input list, not the task (dropped/swapped negatives added). minilm-4: H2 only woke on an appended sentence (woven positives added). minilm-5: 9 in 10 woven answers unusable (bracketed mention, drafted twin); the raw encoder never learnt to match A with B, so the base is now the pair-trained checkpoint of the same network. See MEMORY.md

## 1. Storage and setup

In [2]:
import os, shutil, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/sg-sem"
except Exception as error:  # the VS Code extension may not support Drive mounting
    print("Google Drive not mounted:", error)
    print("Data stays on this runtime only: if it disconnects, the run starts again.")
    ROOT = "/content/sg-sem"
DATA = f"{ROOT}/data-quick" if QUICK_LOOK else f"{ROOT}/data"
RUNS = f"{ROOT}/runs"
os.makedirs(DATA, exist_ok=True); os.makedirs(RUNS, exist_ok=True)
print("data:", DATA)

Mounted at /content/drive
data: /content/drive/MyDrive/sg-sem/data


In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
CODE = "/content/DevScratchPad"
if not os.path.exists(CODE):
    !git clone --depth 1 -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull --ff-only
!git -C {CODE} log --oneline -1
node = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip() if shutil.which("node") else ""
if not node.startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y nodejs > /dev/null 2>&1
!node --version
%cd {CODE}
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!pip install -q -r ml/sg-sem/requirements.txt
PIPE = "npx vite-node scripts/semantic/pipeline.ts"
LIMIT = "--limit 48" if QUICK_LOOK else ""

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
Cloning into '/content/DevScratchPad'...
remote: Enumerating objects: 509, done.
remote: Counting objects: 100% (509/509), done.
remote: Compressing objects: 100% (465/465), done.
remote: Total 509 (delta 16), reused 340 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (509/509), 14.01 MiB | 8.49 MiB/s, done.
Resolving deltas: 100% (16/16), done.
380e367 (grafted, HEAD -> sg-sem, origin/sg-sem) feat(sg-sem): pair-trained base encoder, bracketed weave mention, drafted twin (run msmarco-1)
v20.19.0
/content/DevScratchPad
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 557 packages in 17s
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 148.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Round 1: workflow outlines → seed graphs

In [4]:
!{PIPE} round1 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round1.requests.jsonl --outputs {DATA}/round1.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round1: 200 outline requests → /content/drive/MyDrive/sg-sem/data/round1.requests.jsonl
⠙200 already answered, 0 to go


## 3. Round 2: checker answers, plain-word inputs, gates

In [5]:
!{PIPE} round2 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round2.requests.jsonl --outputs {DATA}/round2.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round2: 171 seeds (157 drafted; rejected {"json":1,"draft":42,"errors":0,"duplicate":0}), 9273 requests {"h1":8190,"h4":455,"h2":628}
why outlines were rejected:
  17 × steps[N].type must be one of orchestrator, worker, verifier, synthesizer, agent, code, approval.
  6 × steps[N].foreach must be <step> or <step>.<key>, e.g. plan.files: the step whose JSON list this step runs over.
  5 × Step … fans out over …, so … must be in its after.
  1 × Step … fans out over …, but … has no … among its keys (laptopN-summary, laptopN-summary, laptopN-summary).
  1 × Step … fans out over …, but … has no … among its keys (length, tone).
  1 × Step … fans out over …, but … has no … among its keys (story).
  1 × Step … fans out over …, but … has no … among its keys (claim_sources).
  1 × Step … fans out over …, but … has no … among its keys (laptop_N_comparison, laptop_N_comparison, laptop_N_comparison).
⠙8700 already answered, 573 to go
tokenizer_config.json: 7.30kB [00:00, 35.2MB/s]
vocab.json: 2.78

## 4. Assemble the labelled examples

In [6]:
!{PIPE} assemble --out {DATA}
print(open(f"{DATA}/stats.json").read())

⠙assemble: 8948 examples from 171 seeds; dropped {"h1-honest_pass-json":404,"h1-honest_pass-no-evidence":249,"h1-honest_pass-length":142,"h1-no_evidence_pass-length":450,"h1-no_evidence_pass-json":159,"h1-no_evidence_pass-has-evidence":8,"h1-resolved_problem_pass-no-evidence":577,"h1-resolved_problem_pass-json":262,"h1-resolved_problem_pass-no-fix":239,"h1-resolved_problem_pass-length":168,"h1-buried_problem_pass-json":293,"h1-buried_problem_pass-not-a-problem":291,"h1-buried_problem_pass-open-count":267,"h1-buried_problem_pass-length":2,"h4-unreadable":17,"h1-buried_number_pass-not-a-problem":257,"h1-buried_number_pass-open-count":238,"h1-buried_number_pass-json":278,"h1-buried_number_pass-length":13,"h1-flag-lost":3,"h2-checker-error":1,"h1-honest_number_pass-json":124,"h1-honest_number_pass-length":1016,"h1-honest_number_pass-no-evidence":83,"h1-honest_number_pass-over-limit":13,"h1-honest_number_pass-open-count":4,"h2-weave-length":43,"h2-weave-off-task":123,"h2-weave-brackets":77,

In [7]:
# For review: raw answers of the text model, two per kind and style (newest first), and the assembled samples.
import json, collections
requests = {json.loads(l)["id"]: json.loads(l) for l in open(f"{DATA}/round2.requests.jsonl")}
shown = collections.Counter()
for line in reversed(open(f"{DATA}/round2.outputs.jsonl").readlines()):
    row = json.loads(line); request = requests.get(row["id"])
    if request is None:  # an answer to an older prompt
        continue
    kind = (request["kind"], request["meta"].get("style", request["meta"].get("mode", "")))
    if shown[kind] < 2:
        shown[kind] += 1
        print(f"--- {' '.join(filter(None, kind))} {request['meta']}")
        print(row["text"][:1200])
        print()
print(open(f"{DATA}/samples.md").read()[:12000])

--- h2 weave {'ref': 'RESEARCH COMPETING PRODUCTS', 'mode': 'weave', 'seed': 'draft-outline-15-3', 'node': 'create-description', 'pred': 'research-competing-products'}
{"task": "Analyze [RESEARCH COMPETING PRODUCTS] to identify strengths and weaknesses, then create a compelling product description.", "task_without": "Create a compelling product description highlighting unique selling points."}

--- h2 weave {'ref': 'RESEARCH KEYWORDS', 'mode': 'weave', 'seed': 'draft-outline-15-3', 'node': 'create-description', 'pred': 'research-keywords'}
{"task": "Craft a product description using [RESEARCH KEYWORDS] and COMPETITIVE ANALYSIS: Highlight unique features, appeal to target audience, and differentiate from [COMPETITORS]. Ensure engaging and accurate.", "task_without": "Craft a product description highlighting unique features, appealing to target audience, and differentiating from competitors. Ensure engaging and accurate."}

--- h1 honest_number_pass {'style': 'honest_number_pass', 'seed'

## 5. Train (runs when `QUICK_LOOK = False`)

In [8]:
if TRAIN:
    !cd ml/sg-sem && python -m sgsem.train --data {DATA}/examples.jsonl --out {RUNS}/{RUN} --base {BASE} --epochs {EPOCHS}
else:
    print("QUICK_LOOK is on: training waits for the full data.")

tokenizer_config.json: 1.33kB [00:00, 7.14MB/s]
vocab.txt: 232kB [00:00, 56.3MB/s]
tokenizer.json: 711kB [00:00, 166MB/s]
special_tokens_map.json: 100% 132/132 [00:00<00:00, 1.08MB/s]
train examples: {'h1': 2056, 'h2': 996, 'h3': 733, 'h4': 2792} dev: {'h1': 362, 'h2': 219, 'h3': 102, 'h4': 424}
config.json: 100% 791/791 [00:00<00:00, 7.20MB/s]
I0000 00:00:1792749611.770493    3788 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792749611.842740    3788 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792749614.355684    3788 port.cc:153] oneDNN

In [9]:
import json
if TRAIN:
    report = json.load(open(f"{RUNS}/{RUN}/metrics.json"))
    print(json.dumps({k: report[k] for k in ["test", "rules_baseline_test", "meets_bar_on_test", "minutes"]}, indent=1))
    for section in open(f"{RUNS}/{RUN}/errors.md").read().split("\n## ")[1:]:  # the worst dev examples, per head
        print("## " + section[:2500])

{
 "test": {
  "h1": {
   "n": 232,
   "temperature": 2.2,
   "precision_at_05": 0.9869281045751634,
   "recall_at_05": 0.9869281045751634,
   "f1_at_05": 0.9869281045751634,
   "auroc": 0.9954496566559113,
   "threshold": 0.5,
   "precision_at_threshold": 0.9869281045751634,
   "recall_at_threshold": 0.9869281045751634,
   "score": 0.9869281045751634,
   "sentence_f1": 0.8
  },
  "h2": {
   "n": 160,
   "temperature": 2.5,
   "precision_at_05": 0.75,
   "recall_at_05": 0.6206896551724138,
   "f1_at_05": 0.679245283018868,
   "auroc": 0.8428533824690708,
   "threshold": 0.8658000230789185,
   "precision_at_threshold": 1.0,
   "recall_at_threshold": 0.13793103448275862,
   "score": 0.679245283018868,
   "span_f1": 0.707399871492825
  },
  "h3": {
   "n": 73,
   "temperature": 2.8000000000000003,
   "precision_at_05": 0.9310344827586207,
   "recall_at_05": 0.8181818181818182,
   "f1_at_05": 0.8709677419354839,
   "auroc": 0.9643939393939394,
   "threshold": 0.7587000131607056,
   "precis

## 6. Gold set (decides what ships)

Hand-written graphs, checker answers and gates in `ml/sg-sem/gold/`, on topics outside the training topics and never touched by the text model. Scored with the run's own thresholds, so the numbers are what the connector would do. A head ships only at precision >= 0.9 here.

In [10]:
import json
!npx vite-node scripts/semantic/gold.ts
!cd ml/sg-sem && python -m sgsem.evaluate --run {RUNS}/{RUN} --data gold/examples.jsonl --out {RUNS}/{RUN}/gold
gold = json.load(open(f"{RUNS}/{RUN}/gold/metrics.json"))
print(json.dumps({k: gold[k] for k in ["metrics", "rules_baseline", "meets_bar"]}, indent=1))
for section in open(f"{RUNS}/{RUN}/gold/errors.md").read().split("\n## ")[1:]:  # the worst gold examples, per head
    print("## " + section[:3500])

⠙gold: 7 graphs, 164 examples → ml/sg-sem/gold/examples.jsonl
{"h1":{"label0":20,"label1":22},"h2":{"label0":31,"label1":28},"h4":{"label0":28,"label1":14,"label2":21}}
⠙WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1792749816.149880    4837 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792749816.221952    4837 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792749818.658670    4837 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floatin

In [11]:
# Which cue does H2 read? Every gold cut positive is scored as is and with one thing changed; the column whose p(cut) moves is the cue.
!cd ml/sg-sem && python -m sgsem.probe --run {RUNS}/{RUN} --data gold/examples.jsonl

h2: 59 gold rows, 28 positives
I0000 00:00:1792749838.212145    5001 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792749838.280957    5001 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792749840.725386    5001 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792749840.725875    5001 cudart_stub.cc:31] Could not find cuda drivers on your

In [ ]:
# Does one side alone give the H2 label away on the generated rows? Classifiers on text A only, text B only, the 16 facts only; test AUROC per negative kind.
!cd ml/sg-sem && python -m sgsem.leak --data {DATA}/examples.jsonl

h2: 988 rows
label kind          rows  no {{ref}} in A  blocks in A  lines in B  A words
0     as written     715             0.00         0.91        1.32       33
0     dropped         55             0.00         0.45        1.15       19
0     kept           109             0.39         0.23        1.37       37
1     cut            109             0.39         0.23        1.11       37
rows whose A has no reference at all: 86, of them positive: 0.50

test AUROC of a train-split classifier that sees one side only (0.5 = the side carries nothing, near 1 = the label is in that side alone):
side                  vs kept     vs dropped  vs as written         vs all
text A only              0.46           0.84           0.90           0.82
text B only              1.00           0.32           0.91           0.91
facts only               1.00           0.52           0.95           0.94
